In [1]:
- 트랜스포머 아키텍쳐 기반 이미지 분류 모델 개발. 트랜스포머와 셀프 어텐션을 구현 
- 트랜스포머 모델과 cnn기반 모델의 성능을 비교함. 정확도, 속도, 매개변수개수 등 지표 고려
- ViT 모델 구현 시 코드 포함. 
- vision transformer (VIT)


SyntaxError: invalid syntax (3032343060.py, line 1)

In [ ]:
import torch
import torch.nn as nn

import torch.nn.functional as F

class SelfAttention(nn.Module):
    def __init__ (self, dim, heads):
        super().__init__()
        self.heads = heads
        self.dim = dim
        self.head_dim = dim // heads
        assert (self.head_dim * heads == dim ) ,"dim must be divisible by heads"
        self.to_keys = nn.Linear(self.head_dim, self.head_dim, bias = False)
        self.to_queries = nn.Linear(self.head_dim, self.head_dim, bias = False)
        self.to_values = nn.Linear(self.head_dim, self.head_dim, bias = False)
        self.to_out = nn.Linear(self.dim, self.dim)
    def forward(self, x):
        # x shape: (batch_size, num_patches, dim)
        batch_size, num_patches, dim = x.shape
        
        # Split into multiple heads
        queries = self.to_queries(x).reshape(batch_size, num_patches,)
        


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SelfAttention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.dim = dim
        self.head_dim = dim // heads
        assert (self.head_dim * heads == dim), "dim must be divisible by heads"

        self.to_keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.to_queries = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.to_values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.to_out = nn.Linear(self.dim, self.dim)

    def forward(self, x):
        # x shape: (batch_size, num_patches, dim)
        batch_size, num_patches, dim = x.shape

        # Split into multiple heads
        queries = self.to_queries(x).reshape(batch_size, num_patches, self.heads, self.head_dim)
        keys = self.to_keys(x).reshape(batch_size, num_patches, self.heads, self.head_dim)
        values = self.to_values(x).reshape(batch_size, num_patches, self.heads, self.head_dim)

        # Compute attention scores
        queries = queries.transpose(1, 2)  # (batch_size, heads, num_patches, head_dim)
        keys = keys.transpose(1, 2)  # (batch_size, heads, num_patches, head_dim)
        values = values.transpose(1, 2)  # (batch_size, heads, num_patches, head_dim)

        # Scaled Dot-Product Attention
        scores = torch.matmul(queries, keys.transpose(-2, -1)) / (self.head_dim ** 0.5) # (batch_size, heads, num_patches, num_patches)
        attention = F.softmax(scores, dim=-1)  # (batch_size, heads, num_patches, num_patches)
        context = torch.matmul(attention, values)  # (batch_size, heads, num_patches, head_dim)

        # Concatenate heads and project to output
        context = context.transpose(1, 2).reshape(batch_size, num_patches, dim) # (batch_size, num_patches, dim)
        out = self.to_out(context) # (batch_size, num_patches, dim)
        return out


class ViT(nn.Module):
    def __init__(self, image_size=32, patch_size=4, num_classes=10, dim=64, depth=6, heads=8):
        super().__init__()
        assert image_size % patch_size == 0, "Image size must be divisible by patch size"
        num_patches = (image_size // patch_size) ** 2
        patch_dim = 3 * patch_size ** 2

        self.patch_embedding = nn.Linear(patch_dim, dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.layers = nn.ModuleList([nn.TransformerEncoderLayer(d_model=dim, nhead=heads) for _ in range(depth)])
        self.to_latent = nn.Identity()
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, num_classes)
        )

    def forward(self, img):
        # img shape: (batch_size, 3, image_size, image_size)
        batch_size, _, image_size, _ = img.shape
        patch_size = self.patch_embedding.weight.shape[1] // 3

        img = img.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size)
        img = img.reshape(batch_size, 3 * patch_size * patch_size, -1).transpose(1, 2)  # (batch_size, num_patches, patch_dim)

        x = self.patch_embedding(img) # (batch_size, num_patches, dim)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1) # (batch_size, 1, dim)
        x = torch.cat((cls_tokens, x), dim=1) # (batch_size, 1+num_patches, dim)

        for layer in self.layers:
            x = layer(x)

        x = x[:, 0]
        x = self.to_latent(x)
        return self.mlp_head(x)


# Example usage
model = ViT()
img = torch.randn(64, 3, 32, 32) # batch size 64
output = model(img)  # output shape: (batch_size, num_classes)
print(output.shape)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (256x768 and 48x64)

: 